# Расчёт нормы калорий и КБЖУ (болезни + стадии жизни)

Считает суточную норму калорий и распределение **Б/Ж/У** + рекомендуемую **клетчатку** с учётом:
- пола, возраста, веса, роста;
- уровня физической активности (с разделением на кардио / силовые / смесь);
- цели — поддержание / снижение / набор веса;
- **нозологической группы** (здоровый, диабет 2 типа, ожирение, ХБП, ССЗ);
- **стадии жизни** (спортсмен, пожилой, беременная, кормящая).

**БЖУ зависит от обоих параметров.** При конфликте белка между болезнью и стадией жизни приоритет у ХБП (защита почек). Рискованные комбинации отмечаются предупреждениями.

**Клетчатка** считается отдельно от калорий (её энергетический вклад пренебрежимо мал) — как самостоятельная рекомендация по норме DRI с поправкой на группу здоровья.

> ⚠️ Расчёт ориентировочный, не заменяет консультацию врача/диетолога.

**Как пользоваться:** заполни ячейку «Данные человека» ниже и запусти все ячейки (`Shift+Enter`).


## 1. Импорты

In [1]:
import sys, os
# Чтобы `import diet` работал из папки notebooks/.
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd

from diet import (UserProfile, calculate, CONDITIONS, LIFE_STAGES,
                  ACTIVITY_LEVELS, GOALS)

pd.set_option('display.unicode.east_asian_width', True)
print('Группы здоровья (condition):', list(CONDITIONS))
print('Стадии жизни (life_stage):', list(LIFE_STAGES))
print('Уровни активности:', list(ACTIVITY_LEVELS))
print('Цели:', list(GOALS))

Группы здоровья (condition): ['healthy', 'diabetes_t2', 'obesity', 'ckd', 'cvd']
Стадии жизни (life_stage): ['default', 'athlete_endurance', 'athlete_strength', 'older_adult', 'pregnant', 'lactating']
Уровни активности: ['sedentary', 'light', 'moderate', 'high', 'very_high', 'light_cardio', 'light_strength', 'light_mixed', 'moderate_cardio', 'moderate_strength', 'moderate_mixed', 'high_cardio', 'high_strength', 'high_mixed']
Цели: ['maintain', 'lose', 'gain']


## 2. Данные человека

**condition:** `healthy` / `diabetes_t2` / `obesity` / `ckd` / `cvd`  
**life_stage:** `default` / `athlete_endurance` / `athlete_strength` / `older_adult` / `pregnant` / `lactating`  
**sex:** `male` / `female`  
**goal:** `maintain` / `lose` / `gain`  
**formula:** `who` / `mifflin`

**activity** — базовые уровни или детальные пресеты по типу нагрузки:

| Базовые | Детальные (тип × частота) |
|---|---|
| `sedentary` | `light_cardio` / `light_strength` / `light_mixed` |
| `light` | `moderate_cardio` / `moderate_strength` / `moderate_mixed` |
| `moderate` | `high_cardio` / `high_strength` / `high_mixed` |
| `high` | |
| `very_high` | |

Кардио сжигает больше за минуту → фактор чуть выше; силовые дают EPOC и мышцы → фактор чуть ниже, но всё равно выше сидячего. Базовые уровни (`light`/`moderate`/`high`) = смесь, как раньше.


In [2]:
# ↓↓↓ ЗАПОЛНИ ПОД СЕБЯ ↓↓↓
profile_data = {
    "sex": "female",        # male / female
    "age": 25,              # полных лет
    "weight": 80,           # кг
    "height": 170,          # см
    "activity": "light_strength",   # см. таблицу выше
    "goal": "maintain",    # maintain / lose / gain
}
condition  = "healthy"      # healthy / diabetes_t2 / obesity / ckd / cvd
life_stage = "default"      # default / athlete_endurance / athlete_strength / older_adult / pregnant / lactating
formula    = "who"          # who / mifflin

profile = UserProfile(**profile_data)


Рекомпозиция

## 3. Расчёт

In [3]:
result = calculate(profile, formula=formula, condition=condition, life_stage=life_stage)

## 4. Сравнение формул BMR

In [4]:
pd.DataFrame(result.comparison()).T

,ВОЗ (Скофилд),Миффлин-Сан Жеор
"BMR, ккал",1667,1576
"Расход за день (TDEE), ккал",2250,2128


## 5. Итог

In [5]:
df = pd.DataFrame.from_dict(result.summary(), orient='index', columns=['Значение'])
df

,Значение
Группа здоровья,Здоровый взрослый
Стадия жизни,Обычный взрослый
Формула BMR,who
"BMR по ВОЗ, ккал",1667
"BMR по Миффлину, ккал",1576
"BMR используемый, ккал",1667
"Расход за день (TDEE), ккал",2250
Уровень активности,Лёгкая: силовые 1-3/нед
Цель,Поддержание веса
"Целевые калории, ккал/день",2250


In [6]:
total_macro_kcal = result.protein_kcal + result.fat_kcal + result.carbs_kcal

print(f"Группа здоровья: {result.condition_label}")
print(f"Стадия жизни: {result.life_stage_label}")
print(f"Формула: {result.formula} · Цель: {result.goal_label} · Активность: {result.activity_label}")
print(f"Целевая калорийность: {round(result.target_kcal)} ккал/день")
print()
print(f"Белки:      {round(result.protein_g):>4} г  ({result.protein_kcal:>4} ккал, {result.protein_kcal/total_macro_kcal*100:>4.1f}%)")
print(f"Жиры:       {round(result.fat_g):>4} г  ({result.fat_kcal:>4} ккал, {result.fat_kcal/total_macro_kcal*100:>4.1f}%)")
print(f"Углеводы:   {round(result.carbs_g):>4} г  ({result.carbs_kcal:>4} ккал, {result.carbs_kcal/total_macro_kcal*100:>4.1f}%)")
print(f"Клетчатка:  {round(result.fiber_g):>4} г  (рекомендация, не входит в калории)")


Группа здоровья: Здоровый взрослый
Стадия жизни: Обычный взрослый
Формула: who · Цель: Поддержание веса · Активность: Лёгкая: силовые 1-3/нед
Целевая калорийность: 2250 ккал/день

Белки:        84 г  ( 338 ккал, 15.0%)
Жиры:         75 г  ( 675 ккал, 30.0%)
Углеводы:    309 г  (1238 ккал, 55.0%)
Клетчатка:    32 г  (рекомендация, не входит в калории)


## 6. Предупреждения и рекомендации

Если выбрана рискованная комбинация (ХБП + спортсмен, беременность + похудение/диабет), она отобразится здесь.

In [7]:
if result.warnings:
    print('⚠️  ПРЕДУПРЕЖДЕНИЯ:')
    for i, w in enumerate(result.warnings, 1):
        print(f'  {i}. {w}')
else:
    print('Предупреждений нет — комбинация безопасная.')
print()
if result.notes:
    print('Рекомендации по выбранной группе/стадии:')
    print(result.notes)

Предупреждений нет — комбинация безопасная.



## 7. Сравнение по стадиям жизни (бонус)

Как меняется БЖУ при одном профиле и группе здоровья в зависимости от стадии жизни.

In [8]:
rows = []
for ls in LIFE_STAGES:
    r = calculate(profile, formula=formula, condition=condition, life_stage=ls)
    rows.append({
        "Стадия жизни": r.life_stage_label,
        "Калории": round(r.target_kcal),
        "Белки, г": round(r.protein_g),
        "Жиры, г": round(r.fat_g),
        "Углеводы, г": round(r.carbs_g),
        "Клетчатка, г": round(r.fiber_g),
        "Предупр.": '⚠' if r.warnings else '',
    })
pd.DataFrame(rows)


,Стадия жизни,Калории,"Белки, г","Жиры, г","Углеводы, г","Клетчатка, г",Предупр.
0,Обычный взрослый,2250,84,75,309,32,
1,Спортсмен на выносливость,2250,104,75,290,32,
2,Спортсмен-силовик / набор массы,2250,144,75,250,32,
3,Пожилой (60+),2250,88,75,306,32,
4,Беременность (2-3 триместр),2590,122,86,331,36,
5,Кормление грудью (лактация),2750,128,92,353,39,
